In [163]:
import torch
import numpy as np
from PIL import Image
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

In [23]:
def polynomial_fun(w, x):
    """
    Evaluates a polynomial function given the weight vector w and an input scalar variable x.
    Args:
        w (torch.Tensor): Weight vector of size (M + 1, 1)
        x (torch.Tensor): Input scalar variables of size (N, 1)
    Returns:
        y (torch.Tensor): Output value of the polynomial function which has size (N, 1)
    """
    M = w.shape[0]
    powers = torch.arange(M, dtype=torch.float32)
    x_powers = torch.pow(x, powers)
    y = torch.matmul(x_powers, w)
    return y

In [93]:
def fit_polynomial_ls(x, t, M):
    """ 
    Implement a least squares solver for fitting polynomial functions using PyTorch's linear algebra modules.
    
    Args:
    - x (torch.Tensor): Input data points of shape (N, 1)
    - t (torch.Tensor): Target values of shape (N, 1)
    - M (int): Polynomial degree
    
    Returns:
    - w_hat (torch.Tensor): Optimum weight vector of shape (M+1, 1)
    """
    x_powers = torch.pow(x, torch.arange(M+1, dtype=torch.float32))
    w_hat = torch.linalg.lstsq(x_powers, t).solution
    return w_hat


In [215]:
def fit_polynomial_sgd(x, t, M, learning_rate, minibatch_size):
    """
    Fits a polynomial function using stochastic minibatch gradient descent.

    Args:
        x (torch.Tensor): Input data points of shape (N, 1)
        t (torch.Tensor): Target values of shape (N, 1)
        M (int): Polynomial degree
        learning_rate (float): Learning rate for gradient descent
        minibatch_size (int): Size of the minibatch

    Returns:
        w_opt (torch.Tensor): Optimum weight vector of shape (M+1, 1)
    """
    num_epochs = 3000
    x_powers = torch.pow(x, torch.arange(M+1, dtype=torch.float32))
    train_data = TensorDataset(x_powers, t)
    model = nn.Linear(M+1, 1, bias=False, dtype=torch.float32) 
    mse_loss = nn.MSELoss() 
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) 
    epochs = []
    losses = []
    # Training loop
    for epoch in range(num_epochs):
        minibatch_data = DataLoader(train_data, batch_size=minibatch_size, shuffle=True)
        for x, y in minibatch_data:
            optimizer.zero_grad()  
            prediction = model(x)  
            loss = mse_loss(prediction, y)  
            loss.backward()  
            optimizer.step() 
        epochs.append(epoch)
        losses.append(loss.item())
        # Print loss every 10 epochs
        if (epoch + 1) % 100 == 0:
            print('Epoch: {} Loss {}'.format(epoch + 1, loss.item()))
    w_opt = model.weight
    return w_opt

In [173]:
# Define weight vector
w = torch.tensor([1, 2, 3], dtype=torch.float32).reshape(3, 1)
w

tensor([[1.],
        [2.],
        [3.]])

In [99]:
# Generate training set
x_train = 40.0 * (torch.rand(20, dtype=torch.float32) - 0.5).reshape(20, 1)
y_train = polynomial_fun(w, x_train)
noise_train = (0.2 * torch.randn(20, dtype=torch.float32)).reshape(20, 1)
t_train = y_train + noise_train

In [89]:
# Generate testing set
x_test = 40.0 * (torch.rand(10, dtype=torch.float32) - 0.5).reshape(10, 1)
y_test = polynomial_fun(w, x_test)
noise_test = 0.2 * torch.randn(10, dtype=torch.float32).reshape(10, 1)
t_test = y_test + noise_test

In [94]:
w_hat_ls = fit_polynomial_ls(x_train, t_train, M=3)

In [220]:
fit_polynomial_sgd(x_train, t_train, 3, 0.01, 5)  # batch_size=25, learning rate=0.25

Epoch: 100 Loss 11157.46875
Epoch: 200 Loss 95.42827606201172
Epoch: 300 Loss 34.94776916503906
Epoch: 400 Loss 20.231582641601562
Epoch: 500 Loss 22.98781394958496
Epoch: 600 Loss 24.58200454711914
Epoch: 700 Loss 13.956167221069336
Epoch: 800 Loss 8.595582962036133
Epoch: 900 Loss 6.476941108703613
Epoch: 1000 Loss 4.044993877410889
Epoch: 1100 Loss 2.0603785514831543
Epoch: 1200 Loss 2.401888608932495
Epoch: 1300 Loss 0.8632366061210632
Epoch: 1400 Loss 1.4150114059448242
Epoch: 1500 Loss 0.7779735922813416
Epoch: 1600 Loss 0.5332270860671997
Epoch: 1700 Loss 0.18879815936088562
Epoch: 1800 Loss 0.17467018961906433
Epoch: 1900 Loss 0.28677135705947876
Epoch: 2000 Loss 0.11756730079650879
Epoch: 2100 Loss 0.03739427775144577
Epoch: 2200 Loss 0.03005378507077694
Epoch: 2300 Loss 0.04850706830620766
Epoch: 2400 Loss 0.06297631561756134
Epoch: 2500 Loss 0.019872542470693588
Epoch: 2600 Loss 0.2748897671699524
Epoch: 2700 Loss 144.50753784179688
Epoch: 2800 Loss 0.6763853430747986
Epoch:

Parameter containing:
tensor([[1.0101e+00, 1.9954e+00, 3.0005e+00, 4.0497e-05]], requires_grad=True)

In [180]:
min(losses)

0.04873322322964668